# 16.2c - YOLO11n occlusal initialization comparison

Matched frontal alignment experiment. Data, validation, seed and training settings match notebook 16.2. Initialization uses verified 16.3 checkpoint.

## Environment

Install the pinned detector package.

In [ ]:
%pip install -q ultralytics==8.3.228

## Validate reviewed dataset

Attach the reviewed dataset containing images, YOLO labels and an explicit review manifest. Verify target box geometry, split membership and complete review status.

In [ ]:
from pathlib import Path
import hashlib, json, shutil
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image, ImageDraw
from ultralytics import YOLO
manifests=list(Path('/kaggle/input').rglob('alignment_reviewed_manifest.csv'))
assert len(manifests)==1, 'Attach a reviewed alignment dataset with alignment_reviewed_manifest.csv'
source=manifests[0].parent
manifest=pd.read_csv(manifests[0],dtype={'sample_id':str})
required={'sample_id','split','group_id','image','label','image_sha256','label_sha256','view_review','diastema_review','crowding_review'}
assert required <= set(manifest.columns)
assert manifest.sample_id.is_unique and set(manifest.split)=={'train','val'}
assert manifest.groupby('group_id').split.nunique().max()==1
assert (manifest.view_review=='frontal').all()
for col in ['diastema_review','crowding_review']:
    assert manifest[col].isin(['present','absent']).all()
target=Path('/tmp/alignment_reviewed')
for row in manifest.itertuples():
    for kind,rel,expected in [('images',row.image,row.image_sha256),('labels',row.label,row.label_sha256)]:
        path=(source/rel).resolve()
        assert path.is_relative_to(source.resolve())
        assert hashlib.sha256(path.read_bytes()).hexdigest()==expected
        dest=target/kind/row.split/(row.sample_id+path.suffix)
        dest.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(path,dest)
    rows=(target/'labels'/row.split/(row.sample_id+Path(row.label).suffix)).read_text().splitlines()
    classes=set()
    for line in rows:
        cls,x,y,w,h=map(float,line.split());assert cls in (0,1)
        assert w>0 and h>0 and 0<=x-w/2<x+w/2<=1 and 0<=y-h/2<y+h/2<=1
        classes.add(int(cls))
    assert (0 in classes)==(row.diastema_review=='present')
    assert (1 in classes)==(row.crowding_review=='present')
(target/'data.yaml').write_text(f'path: {target}\ntrain: images/train\nval: images/val\nnames:\n  0: upper_midline_diastema\n  1: anterior_crowding\n')
print('Manifest SHA-256',hashlib.sha256(manifests[0].read_bytes()).hexdigest())
display(manifest.groupby('split')[['diastema_review','crowding_review']].value_counts())
assert torch.cuda.is_available()
print('GPU',torch.cuda.get_device_name(0))

## Visualize a training example

Display box geometry and class names before training.

In [ ]:
row=manifest[manifest.split=='train'].iloc[0]
image=Image.open(target/'images/train'/f"{row.sample_id}{Path(row.image).suffix}").convert('RGB')
draw=ImageDraw.Draw(image)
for line in (target/'labels/train'/f"{row.sample_id}{Path(row.label).suffix}").read_text().splitlines():
    cls,x,y,w,h=map(float,line.split())
    draw.rectangle(((x-w/2)*image.width,(y-h/2)*image.height,(x+w/2)*image.width,(y+h/2)*image.height),outline=['yellow','cyan'][int(cls)],width=3)
plt.figure(figsize=(10,6));plt.imshow(image);plt.axis('off');plt.show()

## Train the comparison

Use verified 16.3 checkpoint, seed 42, 640 input and periodic checkpoints. The remaining settings match notebook 16.2.

In [ ]:
import time
started=time.monotonic()
checkpoints=list(Path('/kaggle/input').rglob('best.pt'))
matches=[path for path in checkpoints if hashlib.sha256(path.read_bytes()).hexdigest()=='30fc101bdad08d5b34b1dd83082e489b30f6c0130f5429d9313a9bbf0ac8b22f']
assert len(matches)==1, 'Attach the verified 16.3 best checkpoint'
model=YOLO(matches[0])
def time_guard(trainer):
    if time.monotonic()-started>5*3600:
        trainer.stop=True
model.add_callback('on_train_epoch_end',time_guard)
result=model.train(data=str(target/'data.yaml'),epochs=60,patience=0,imgsz=640,batch=16,device=0,workers=2,seed=42,deterministic=True,save=True,save_period=10,plots=True,project='/kaggle/working/runs',name='frontal_alignment_c')
print(result.results_dict)

## Inspect results

Show learning curves and save the best-checkpoint fingerprint.

In [ ]:
run=Path(result.save_dir)
history=pd.read_csv(run/'results.csv');history.columns=history.columns.str.strip()
display(history.tail())
history.plot(x='epoch',y=['metrics/mAP50(B)','metrics/mAP50-95(B)'],grid=True);plt.show()
print('best.pt SHA-256',hashlib.sha256((run/'weights/best.pt').read_bytes()).hexdigest())
(run/'metrics.json').write_text(json.dumps(result.results_dict,indent=2))